# 🚨 Early Warning System for Student Dropout Prediction with Explainable AI

## FlexiSAF — Generative AI & Data Science (Advanced) Curriculum

---

## Introduction

As an EdTech company, one of the most impactful applications of machine learning is identifying students at risk of dropping out **before** it happens. An **Early Warning System (EWS)** uses historical academic and demographic data to predict future student outcomes, enabling proactive intervention.

In this notebook, we will:
1. **Explore** the UCI "Predict Students' Dropout and Academic Success" dataset
2. **Build** an XGBoost classifier to predict student dropout vs. graduation
3. **Evaluate** using Precision, Recall, F1-Score, and a Confusion Matrix
4. **Explain** predictions using **SHAP** (SHapley Additive exPlanations) — so teachers know *why* a student is flagged, not just *that* they are

> **Why Explainability Matters:** If a system tells a teacher "Student A is at 85% risk," the immediate question is **"Why?"** SHAP answers that question with concrete, feature-level explanations — enabling personalized, data-driven interventions.

### Dataset Source
- **UCI Machine Learning Repository**: [Predict Students' Dropout and Academic Success](https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success)
- 4,424 students with 36 features covering demographics, socioeconomic background, and academic performance

In [ ]:
# ============================================================
# CELL 1: IMPORTS & CONFIGURATION
# ============================================================

# --- Data manipulation ---
import pandas as pd
import numpy as np

# --- Visualization ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Scikit-Learn: preprocessing & evaluation ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score, roc_auc_score
)

# --- XGBoost: Gradient Boosting Ensemble ---
from xgboost import XGBClassifier

# --- SHAP: Explainable AI ---
import shap

# --- Suppress warnings for cleaner output ---
import warnings
warnings.filterwarnings('ignore')

# --- Plotting aesthetics & reproducibility ---
%matplotlib inline
sns.set_style('whitegrid')
np.random.seed(42)

print("All libraries loaded successfully. ✅")

---
## 1. Data Loading & Initial Exploration

The dataset uses a **semicolon** (`;`) separator and has some quirks in column names (e.g., tab characters). We'll clean those up during loading.

In [ ]:
# ============================================================
# CELL 2: LOAD AND INSPECT THE DATA
# ============================================================

# Load dataset with semicolon separator
df = pd.read_csv('data.csv', sep=';')

# Clean column names: strip whitespace and tab characters
# The original CSV has a tab character in 'Daytime/evening attendance'
df.columns = df.columns.str.strip().str.replace('\t', '', regex=False)

# --- Basic info ---
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumn names ({len(df.columns)} total):")
for i, col in enumerate(df.columns):
    print(f"  {i:2d}. {col}")

print("\n" + "=" * 60)
print("FIRST 5 ROWS")
print("=" * 60)
display(df.head())

# --- Data types and info ---
print("\n" + "=" * 60)
print("DATA TYPES")
print("=" * 60)
df.info()

# --- Summary statistics ---
print("\n" + "=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)
display(df.describe())

# --- Missing values check ---
missing = df.isnull().sum()
total_missing = missing.sum()
if total_missing == 0:
    print("\n✅ No missing values found in the dataset.")
else:
    print(f"\n⚠️ Total missing values: {total_missing}")
    print(missing[missing > 0])

---
## 2. Exploratory Data Analysis (EDA)

Before building a model, it's essential to understand the data. We'll examine:
1. The **distribution of the target variable** (Dropout / Graduate / Enrolled)
2. **Feature correlations** to identify the strongest predictors
3. **Key academic metrics** across outcome categories

In [ ]:
# ============================================================
# EDA PLOT 1: Target Variable Distribution
# ============================================================

plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df, x='Target', order=df['Target'].value_counts().index,
                   palette='viridis')
plt.title('Distribution of Student Outcomes', fontsize=14)
plt.xlabel('Status', fontsize=12)
plt.ylabel('Count', fontsize=12)

# Annotate bars with counts and percentages
total = len(df)
for p in ax.patches:
    count = int(p.get_height())
    pct = f'{100 * count / total:.1f}%'
    ax.annotate(f'{count}\n({pct})',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=11,
                xytext=(0, 5), textcoords='offset points')

plt.tight_layout()
plt.show()

print("\nTarget value counts:")
print(df['Target'].value_counts())

**📊 Insight:** The dataset has three classes:
- **Graduate** (~50%) — successfully completed the program
- **Dropout** (~32%) — left before completing
- **Enrolled** (~18%) — still in the program (outcome unknown)

Since 'Enrolled' students have no definitive outcome yet, we'll **exclude them** and focus on a clear binary prediction: **Dropout vs. Graduate**.

In [ ]:
# ============================================================
# EDA PLOT 2: Correlation Heatmap (Top 15 Features)
# ============================================================

# Temporarily encode the target to compute correlations
temp_df = df.copy()
temp_df['Target_Encoded'] = temp_df['Target'].map(
    {'Dropout': 0, 'Enrolled': 1, 'Graduate': 2}
)

# Find the top 15 features most correlated with the target
correlations = temp_df.corr(numeric_only=True)['Target_Encoded'].abs()
correlations = correlations.sort_values(ascending=False)
top_features = correlations.index[1:16]  # Exclude target itself

print("Top 15 features most correlated with student outcome:")
for i, feat in enumerate(top_features, 1):
    print(f"  {i:2d}. {feat:50s} (|r| = {correlations[feat]:.3f})")

# Plot the heatmap of these top features
plt.figure(figsize=(12, 9))
sns.heatmap(
    temp_df[top_features].corr(),
    annot=True, fmt='.2f', cmap='coolwarm',
    square=True, cbar_kws={'shrink': 0.8}
)
plt.title('Correlation Heatmap — Top 15 Predictive Features', fontsize=14)
plt.tight_layout()
plt.show()

**📊 Insight:** Curricular units in both semesters (especially **approved units** and **grades**) dominate the correlation rankings. This makes intuitive sense: students who pass more courses with higher grades are far less likely to drop out. These will be the model's strongest signals.

In [ ]:
# ============================================================
# EDA PLOT 3: Distribution of Key Academic Features by Outcome
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot A: Admission Grade
sns.histplot(data=df, x='Admission grade', hue='Target',
             kde=True, palette='Set2', bins=30, ax=axes[0])
axes[0].set_title('Distribution of Admission Grade by Outcome', fontsize=12)

# Plot B: 1st Semester Approved Units
sns.histplot(data=df, x='Curricular units 1st sem (approved)', hue='Target',
             kde=True, palette='Set2', bins=15, multiple='stack', ax=axes[1])
axes[1].set_title('1st Semester Approved Units by Outcome', fontsize=12)

plt.tight_layout()
plt.show()

**📊 Insight:** Students who drop out tend to have **fewer approved curricular units** in the 1st semester. Admission grade shows more overlap between groups — it's a weaker predictor on its own compared to first-semester performance.

In [ ]:
# ============================================================
# EDA PLOT 4: Box Plots of Key Features by Outcome
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1st Semester Grades
sns.boxplot(data=df, x='Target', y='Curricular units 1st sem (grade)',
            palette='Set3', ax=axes[0])
axes[0].set_title('1st Semester Grades', fontsize=12)

# 2nd Semester Grades
sns.boxplot(data=df, x='Target', y='Curricular units 2nd sem (grade)',
            palette='Set3', ax=axes[1])
axes[1].set_title('2nd Semester Grades', fontsize=12)

# Age at Enrollment
sns.boxplot(data=df, x='Target', y='Age at enrollment',
            palette='Set3', ax=axes[2])
axes[2].set_title('Age at Enrollment', fontsize=12)

plt.tight_layout()
plt.show()

**📊 Insight:** 
- Dropouts have **significantly lower median grades** in both semesters — a clear academic distress signal
- Older students at enrollment appear slightly more prone to dropping out, possibly due to competing life responsibilities (work, family)
- These patterns validate that an ML model can learn meaningful distinctions between successful and at-risk students

---
## 3. Data Preprocessing

Before training the model, we need to:

| Step | What | Why |
|---|---|---|
| 1 | Remove 'Enrolled' students | Their outcome is unknown — can't train on them |
| 2 | Encode Target | Dropout = **1** (at-risk, the class we want to detect) |
| 3 | Train/Test split (80/20) | Evaluate on unseen data, stratified to keep class ratio |
| 4 | Feature scaling | StandardScaler for consistent feature ranges |

In [ ]:
# ============================================================
# CELL: DATA PREPROCESSING
# ============================================================

# Step 1: Filter to binary classification — remove 'Enrolled' students
df_binary = df[df['Target'] != 'Enrolled'].copy()
print(f"After removing 'Enrolled': {len(df_binary):,} students remain")
print(f"  - Graduate: {(df_binary['Target'] == 'Graduate').sum()}")
print(f"  - Dropout:  {(df_binary['Target'] == 'Dropout').sum()}")

# Step 2: Encode Target → Dropout=1 (At-Risk), Graduate=0 (Safe)
df_binary['Target'] = df_binary['Target'].map({'Dropout': 1, 'Graduate': 0})

# Separate features (X) and target (y)
X = df_binary.drop(columns=['Target'])
y = df_binary['Target']

# Step 3: Train/Test Split (80/20, stratified, reproducible)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Step 4: Feature Scaling
# Note: Tree-based models like XGBoost don't strictly require scaling,
# but scaling ensures consistency and is good practice for pipeline portability.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)      # Fit on train, transform train
X_test_scaled  = scaler.transform(X_test)            # Only transform test (no fit!)

# Keep as DataFrames to preserve feature names (needed for SHAP plots)
feature_names = X.columns.tolist()
X_train_df = pd.DataFrame(X_train_scaled, columns=feature_names)
X_test_df  = pd.DataFrame(X_test_scaled,  columns=feature_names)

print(f"\nTraining set: {X_train_df.shape[0]} samples")
print(f"Test set:     {X_test_df.shape[0]} samples")
print(f"Features:     {X_train_df.shape[1]}")

---
## 4. Model Training — XGBoost (Ensemble Learning)

### Why XGBoost?
**XGBoost** (eXtreme Gradient Boosting) is an advanced **ensemble algorithm** that builds hundreds of decision trees sequentially, where each new tree corrects the errors of the previous ones. It's the go-to choice for tabular data because:

- ✅ Handles class imbalance via `scale_pos_weight`
- ✅ Built-in regularization prevents overfitting
- ✅ Natively provides feature importance
- ✅ Compatible with SHAP's `TreeExplainer` for fast explanations

> **Alternative Ensemble Methods:**
> - **Random Forest** — trains trees in parallel (bagging) rather than sequentially (boosting). Simpler but often slightly less accurate.
> - **Stacking/Voting Ensemble** — combines XGBoost, Random Forest, and Logistic Regression. Can squeeze out extra performance in production.
> - **LightGBM / CatBoost** — other gradient boosting libraries with different optimization strategies.

In [ ]:
# ============================================================
# CELL: TRAIN XGBOOST MODEL
# ============================================================

# Calculate class imbalance ratio for scale_pos_weight
# This tells XGBoost to pay MORE attention to the minority class (Dropout)
n_graduate = (y_train == 0).sum()  # Majority class
n_dropout  = (y_train == 1).sum()  # Minority class (what we want to detect)
imbalance_ratio = n_graduate / n_dropout

print(f"Class distribution in training set:")
print(f"  Graduate (0): {n_graduate} ({100 * n_graduate / len(y_train):.1f}%)")
print(f"  Dropout  (1): {n_dropout}  ({100 * n_dropout / len(y_train):.1f}%)")
print(f"  Imbalance ratio (scale_pos_weight): {imbalance_ratio:.2f}")

# Initialize and configure the XGBoost classifier
xgb_model = XGBClassifier(
    n_estimators=200,              # Number of boosting rounds (trees)
    max_depth=6,                   # Maximum tree depth (controls complexity)
    learning_rate=0.1,             # Step size shrinkage (lower = more robust)
    scale_pos_weight=imbalance_ratio,  # Compensate for class imbalance
    random_state=42,               # Reproducibility
    use_label_encoder=False,       # Suppress deprecation warning
    eval_metric='logloss'          # Binary cross-entropy for evaluation
)

# Train the model
print("\nTraining XGBoost model...")
xgb_model.fit(X_train_df, y_train)
print("✅ Model training complete.")

---
## 5. Model Evaluation

For an **Early Warning System**, understanding evaluation metrics in the educational context is critical:

| Metric | What it measures | In our context |
|---|---|---|
| **Precision** (Dropout) | Of all students *flagged* as at-risk, how many actually dropped out? | High precision → fewer false alarms (less wasted intervention effort) |
| **Recall** (Dropout) | Of all students who *actually* dropped out, how many did we catch? | High recall → fewer students slip through the cracks |
| **F1-Score** | Harmonic mean of Precision & Recall | Balances both concerns |

> **⚠️ In early warning, Recall for Dropout is the most critical metric.** A false positive (flagging a student who would have graduated) results in unnecessary but harmless extra support. A false negative (missing a student who actually drops out) is a missed opportunity to intervene.

In [ ]:
# ============================================================
# CELL: MODEL EVALUATION
# ============================================================

# Generate predictions on the test set
y_pred = xgb_model.predict(X_test_df)
y_pred_proba = xgb_model.predict_proba(X_test_df)[:, 1]  # Probability of Dropout

# --- Classification Report ---
print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(
    y_test, y_pred,
    target_names=['Graduate (Safe)', 'Dropout (At-Risk)']
))

# --- Key metrics for the Dropout class ---
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_pred_proba)

print("=" * 60)
print("DROPOUT (AT-RISK) CLASS METRICS")
print("=" * 60)
print(f"  Precision : {prec:.3f}  (of flagged students, {prec*100:.1f}% actually dropped out)")
print(f"  Recall    : {rec:.3f}  (caught {rec*100:.1f}% of actual dropouts)")
print(f"  F1-Score  : {f1:.3f}")
print(f"  ROC-AUC   : {auc:.3f}")

# --- Confusion Matrix ---
plt.figure(figsize=(7, 5))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Graduate', 'Dropout']
)
disp.plot(cmap='Blues', values_format='d', ax=plt.gca())
plt.title('Confusion Matrix', fontsize=14)
plt.grid(False)  # Cleaner look without grid
plt.tight_layout()
plt.show()

# Explain the quadrants
tn, fp, fn, tp = cm.ravel()
print(f"\nConfusion Matrix Breakdown:")
print(f"  ✅ True Negatives  (correctly identified Safe):     {tn}")
print(f"  ⚠️  False Positives (flagged but actually Safe):    {fp}")
print(f"  ❌ False Negatives (MISSED at-risk students):       {fn}")
print(f"  ✅ True Positives  (correctly caught at-risk):      {tp}")

---
## 6. SHAP Explainability — Making the Model Transparent

### What is SHAP?
**SHAP** (SHapley Additive exPlanations) uses game theory to assign each feature an "importance value" for each individual prediction. It answers:
- **Globally**: Which features matter most *overall*?
- **Locally**: For *this specific student*, why was the prediction Dropout?

This is essential for FlexiSAF because:
- Teachers need to understand *why* a student is flagged to take the right action
- Parents need clear, transparent explanations
- School policies require accountability and fairness in automated decisions

In [ ]:
# ============================================================
# CELL: SHAP — GLOBAL FEATURE IMPORTANCE
# ============================================================

# Initialize SHAP's TreeExplainer (optimized for tree-based models)
explainer = shap.TreeExplainer(xgb_model)

# Compute SHAP values for all test samples
# This returns an Explanation object with .values, .base_values, .data
shap_values = explainer(X_test_df)

print(f"SHAP values computed for {len(X_test_df)} test samples.")
print(f"Shape: {shap_values.values.shape}")

# --- SHAP Bar Plot: Global Feature Importance ---
# Shows which features have the largest average impact on predictions
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test_df, plot_type='bar', show=False)
plt.title('Global Feature Importance (SHAP Bar Plot)', fontsize=14)
plt.tight_layout()
plt.show()

**📊 Interpretation of the Bar Plot:**
The features at the top have the **greatest overall impact** on the model's predictions. For a school administrator, this reveals which data points are most valuable to collect and monitor — focus on the top features when designing intervention programs.

In [ ]:
# ============================================================
# CELL: SHAP DOT PLOT — Directional Feature Impact
# ============================================================

# The dot plot shows HOW each feature affects predictions:
# - Each dot = one student
# - X-axis = SHAP value (positive → pushes toward Dropout, negative → toward Graduate)
# - Color = feature value (red = high, blue = low)

plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test_df, show=False)
plt.title('Feature Impact Direction (SHAP Dot Plot)', fontsize=14)
plt.tight_layout()
plt.show()

**📊 How to read the Dot Plot:**
- **Red dots on the right** = a *high* feature value pushes prediction toward Dropout
- **Blue dots on the right** = a *low* feature value pushes prediction toward Dropout

For example, if "Curricular units 2nd sem (approved)" shows blue dots (low values) clustered on the right (positive SHAP), it means: *students who approved few 2nd-semester units are strongly predicted as dropouts*. This is actionable: teachers should watch for students failing multiple courses.

In [ ]:
# ============================================================
# CELL: SHAP WATERFALL PLOT — Explaining a Single Student
# ============================================================

# Find a student who was correctly identified as Dropout (True Positive)
# This makes the explanation most compelling
tp_indices = np.where((y_test.values == 1) & (y_pred == 1))[0]
student_idx = tp_indices[0]  # Pick the first true positive

# Get the model's predicted probability for this student
student_prob = y_pred_proba[student_idx]

print(f"=" * 60)
print(f"INDIVIDUAL STUDENT EXPLANATION")
print(f"=" * 60)
print(f"Student index in test set: {student_idx}")
print(f"Actual outcome:     DROPOUT")
print(f"Predicted outcome:  DROPOUT (probability: {student_prob:.1%})")
print(f"\nThe waterfall plot below shows WHY this student was flagged:")

# Generate waterfall plot
plt.figure(figsize=(10, 7))
shap.plots.waterfall(shap_values[student_idx], show=False)
plt.title(f'Why Was This Student Flagged as At-Risk? (P(Dropout) = {student_prob:.1%})',
          fontsize=13)
plt.tight_layout()
plt.show()

**📊 How to read the Waterfall Plot:**
- The plot starts from the **base value** (average prediction for all students)
- Each row shows a feature and how it pushed the prediction **up** (red, toward Dropout) or **down** (blue, toward Graduate)
- The **f(x)** value at the top is the final model output for this student

**What a teacher/counselor can learn:** Instead of just seeing "Student A is at 85% risk," they can see: *"This student is at risk primarily because they approved 0 curricular units in the 2nd semester, had low 1st-semester grades, and their tuition fees are not up to date."* This enables targeted, empathetic conversations.

---
## 7. FlexiSAF Integration Recommendations

Integrating this Early Warning System into FlexiSAF's **SAFSIMS** platform can transform how schools manage student retention:

### 🔄 Real-time Risk Scoring
Deploy the model as an API endpoint. As semester grades or attendance records are updated in SAFSIMS, the system automatically recalculates each student's risk score in real time.

### 📊 Teacher Dashboard with SHAP Explanations
Build a module in the teacher portal that:
- Highlights at-risk students with a color-coded risk gauge
- Shows the **top 3 reasons** why each student was flagged (derived from SHAP waterfall data)
- Example display: *"⚠️ Flagged primarily due to: (1) Low 1st-semester grades, (2) Tuition fees not current, (3) High number of unapproved units"*

### 📱 Automated Alerts
- Send SMS/email alerts to **guidance counselors** when a student crosses a critical risk threshold (e.g., >70% dropout probability)
- Send gentler notifications to **parents** encouraging engagement with the school

### 📅 Mid-Semester Checkpoints
Run batch predictions at **mid-semester** using continuous assessment scores. This gives teachers a 6-8 week window to intervene before final exams.

### ⚖️ Ethical & Privacy Considerations
- **Data Privacy**: Anonymize data in analytics dashboards. Comply with local data protection regulations.
- **Algorithmic Bias**: Regularly audit the model for bias against demographic groups (gender, nationality, socioeconomic status).
- **Human-in-the-Loop**: The model is an *advisor*, not an absolute judge. Teachers must apply context, empathy, and professional judgment alongside the model's recommendations.
- **Transparency**: Students and parents should know the system exists and have the right to understand or challenge their risk assessment.

---
## 8. Conclusion

In this notebook, we:

1. ✅ **Explored** the UCI student dataset — identifying curricular performance as the strongest predictor of dropout
2. ✅ **Built** an XGBoost ensemble model that predicts dropout with strong Recall (catching most at-risk students)
3. ✅ **Evaluated** with Precision, Recall, F1-Score, ROC-AUC, and a Confusion Matrix
4. ✅ **Explained** predictions using SHAP — both globally (which features matter most) and locally (why a specific student was flagged)

### Key Takeaway for FlexiSAF
By combining **accurate prediction** (ensemble learning) with **transparent explanations** (SHAP), this system moves beyond a black-box model into a tool that educators can trust, understand, and act on. Integrating it into SAFSIMS would position FlexiSAF as a leader in data-driven student support.